# 01 — 下载 RQData 商品期货原始行情

本 Notebook 只负责：初始化 RQData、保存期货元数据、筛选商品品种、下载比例复权主力连续和可选的 99 指数连续。

本 Notebook 不计算收益率、波动率、相关性或市场状态。

In [ ]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "config" / "config.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import cta_research  # noqa: E402
from cta_research import load_config  # noqa: E402

assert Path(cta_research.__file__).resolve().is_relative_to((PROJECT_ROOT / "src").resolve())

config = load_config(PROJECT_ROOT / "config" / "config.yaml")
START_DATE = config["start_date"]
END_DATE = config["end_date"] or date.today().isoformat()
DOWNLOAD_INDEX_99 = bool(config["rqdata"]["download_index_99"])
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
paths = {
    name: PROJECT_ROOT / config["paths"][name]
    for name in ["future_instruments", "dominant_prices", "index_99_prices"]
}
cache_status = {name: paths[name].exists() for name in paths}
display(
    pd.Series(
        {
            **config,
            "DATA_DIR": str(DATA_DIR),
            "DOWNLOAD_INDEX_99": DOWNLOAD_INDEX_99,
            "RQDATA_END_DATE": END_DATE,
            "CACHE_STATUS": cache_status,
        },
        name="value",
    ).to_frame()
)
print("Existing caches will be incrementally updated.")

## 1. 初始化 RQData

In [ ]:
import rqdatac
from rqdatac import futures

rqdatac.init()
print("RQData initialized; credentials are not displayed or saved.")

## 2. 获取期货合约元数据

In [ ]:
future_instruments = rqdatac.all_instruments(type="Future")
if not isinstance(future_instruments, pd.DataFrame) or future_instruments.empty:
    raise ValueError("RQData returned no Future metadata.")
future_instruments.to_parquet(paths["future_instruments"], index=False)
print("Rows:", len(future_instruments))
print("Columns:", future_instruments.columns.tolist())
display(future_instruments.head())
display(future_instruments["exchange"].value_counts(dropna=False).rename("contracts").to_frame())

## 3. 筛选商品期货品种

In [ ]:
from cta_research import select_commodity_instruments

commodity_instruments, excluded_instruments = select_commodity_instruments(
    future_instruments,
    set(config["commodity_exchanges"]),
)
commodity_symbols = sorted(commodity_instruments["underlying_symbol"].dropna().astype(str).str.upper().unique())
print("商品期货品种数量：", len(commodity_symbols))
print("商品期货品种：", commodity_symbols)
display(commodity_instruments.groupby("exchange")["underlying_symbol"].nunique().rename("symbols").to_frame())
display(excluded_instruments.groupby("exclusion_reason").size().rename("rows").to_frame())

## 4. 下载比例复权主力连续日行情

In [ ]:
import time

from cta_research import (
    group_incremental_requests,
    merge_rqdata_price_updates,
    normalize_rqdata_prices,
)


def batches(values: list[str], size: int = 10):
    for start in range(0, len(values), size):
        yield values[start : start + size]


def retry(function, *args, **kwargs):
    error = None
    for attempt in range(1, 4):
        try:
            return function(*args, **kwargs)
        except Exception as exception:
            error = exception
            if attempt < 3:
                time.sleep(attempt)
    raise error


dominant_prices = pd.read_parquet(paths["dominant_prices"]) if paths["dominant_prices"].exists() else pd.DataFrame()
dominant_request_groups = group_incremental_requests(
    commodity_symbols, dominant_prices, START_DATE, END_DATE
)
dominant_failures = []
dominant_parts = []
for request_start, symbols in dominant_request_groups.items():
    rqdata_parameters = {
        "start_date": request_start,
        "end_date": END_DATE,
        "frequency": config["rqdata"]["frequency"],
        "fields": ["close"],
        "adjust_type": config["rqdata"]["adjust_type"],
        "adjust_method": config["rqdata"]["adjust_method"],
        "rule": config["rqdata"]["rule"],
        "rank": config["rqdata"]["rank"],
    }
    for symbol_batch in batches(symbols):
        try:
            result = retry(futures.get_dominant_price, symbol_batch, **rqdata_parameters)
            if isinstance(result, pd.DataFrame) and not result.empty:
                dominant_parts.append(normalize_rqdata_prices(result))
        except Exception:
            for symbol in symbol_batch:
                try:
                    result = retry(futures.get_dominant_price, symbol, **rqdata_parameters)
                    if isinstance(result, pd.DataFrame) and not result.empty:
                        dominant_parts.append(normalize_rqdata_prices(result))
                    else:
                        dominant_failures.append({"underlying_symbol": symbol, "error": "empty response"})
                except Exception as exception:
                    dominant_failures.append(
                        {"underlying_symbol": symbol, "error": f"{type(exception).__name__}: {exception}"}
                    )
dominant_updates = pd.concat(dominant_parts, ignore_index=True) if dominant_parts else pd.DataFrame()
if not dominant_updates.empty:
    dominant_prices = merge_rqdata_price_updates(dominant_prices, dominant_updates)
    dominant_prices.to_parquet(paths["dominant_prices"], index=False)
elif dominant_prices.empty:
    raise RuntimeError("No dominant price data was returned.")
print("主力连续行数：", len(dominant_prices))
print("成功品种数：", dominant_prices["underlying_symbol"].nunique())
print("主力连续新增行数：", len(dominant_updates))
print("主力连续请求起点：", dominant_request_groups)
display(pd.DataFrame(dominant_failures))
display(dominant_prices.head())

## 5. 可选下载 99 指数连续

In [ ]:
index_99_prices = (
    pd.read_parquet(paths["index_99_prices"])
    if paths["index_99_prices"].exists()
    else pd.DataFrame()
)
index_99_failures = []
index_99_parts = []
index_99_symbols = [f"{symbol}99" for symbol in commodity_symbols]
index_99_request_groups = group_incremental_requests(
    index_99_symbols, index_99_prices, START_DATE, END_DATE
)
if DOWNLOAD_INDEX_99:
    for request_start, symbols in index_99_request_groups.items():
        for id_batch in batches(symbols):
            request_parameters = {
                "start_date": request_start,
                "end_date": END_DATE,
                "frequency": config["rqdata"]["frequency"],
                "fields": ["close"],
                "adjust_type": "none",
            }
            try:
                result = retry(rqdatac.get_price, id_batch, **request_parameters)
                if isinstance(result, pd.DataFrame) and not result.empty:
                    index_99_parts.append(normalize_rqdata_prices(result, symbol_column="order_book_id"))
            except Exception:
                for order_book_id in id_batch:
                    try:
                        result = retry(rqdatac.get_price, order_book_id, **request_parameters)
                        if isinstance(result, pd.DataFrame) and not result.empty:
                            index_99_parts.append(normalize_rqdata_prices(result, symbol_column="order_book_id"))
                        else:
                            index_99_failures.append({"order_book_id": order_book_id, "error": "empty response"})
                    except Exception as exception:
                        index_99_failures.append(
                            {"order_book_id": order_book_id, "error": f"{type(exception).__name__}: {exception}"}
                        )
    index_99_updates = pd.concat(index_99_parts, ignore_index=True) if index_99_parts else pd.DataFrame()
    if not index_99_updates.empty:
        index_99_prices = merge_rqdata_price_updates(index_99_prices, index_99_updates)
        index_99_prices.to_parquet(paths["index_99_prices"], index=False)
        print("99指数连续行数：", len(index_99_prices))
        print("99指数连续新增行数：", len(index_99_updates))
    elif index_99_prices.empty:
        print("99指数连续没有返回数据。")
    print("99指数连续请求起点：", index_99_request_groups)
else:
    index_99_updates = pd.DataFrame()
    print("DOWNLOAD_INDEX_99=False; skipped.")
display(pd.DataFrame(index_99_failures))

## 6. 下载完成

In [ ]:
print("RQData 原始行情已保存。")
print("主力连续：", paths["dominant_prices"])
if DOWNLOAD_INDEX_99:
    print("99 指数连续：", paths["index_99_prices"] if paths["index_99_prices"].exists() else "无返回数据")